# Limpieza de datos

La mayoría de la gente piensa que ser científico de datos significa estar corriendo modelos avanzados de Machine Learning todo el tiempo. La realidad es muy distinta:

![Tiempo de un científico de datos](./img/datascientist_time.jpeg)

La gran mayoría del tiempo se va **limpiando y organizando** los datos con los que queremos trabajar.

Los científicos de datos utilizamos herramientas como Pandas, Numpy, Matplotlib y Seaborn para limpiar los datos con los que queremos hacer algo.

## ETL

Un tipo de tarea que realizamos con gran frecuencia los científicos de datos son los **ETL**.

![Proceso de ETL](./img/etl.png)

ETL es un acrónimo que significa Extract, Transform, Load. Es un proceso que se utiliza para extraer datos de una fuente, transformarlos en un formato que sea adecuado para el análisis y cargarlos en una base de datos o algún otro sistema de almacenamiento.

## Ejemplo práctico

Los datos que usaremos para esta limpieza y nuestro siguiente análisis son datos de incidencia delictiva en nuestro país.

La iniciativa de datos abiertos del gobierno de México nos proporciona datos de incidencia delictiva desde 2015 hasta la fecha. Los datos se actualizan todos los meses y se pueden descargar desde el siguiente enlace: https://www.gob.mx/sesnsp/acciones-y-programas/datos-abiertos-de-incidencia-delictiva

---

Como podemos ver en el portal, se proporcionan los datos tanto a nivel estatal como a nivel municipal. En este caso, utilizaremos los datos a nivel estatal.

Descarguemos los datos y guardemos el archivo CSV en la carpeta data con el nombre `datos_delitos.csv`.

---

Ahora leamos el archivo CSV y veamos cómo se ven los datos.

Primero que nada, importemos pandas

In [1]:
import pandas as pd

In [15]:
df = pd.read_csv('data/datos_delitos.csv')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf1 in position 1: invalid continuation byte

Aquí tenemos un error muy común que suele ocurrir cuando un archivo se guarda en una computadora con cierto "encoding". 

Un encoding es una tabla que relaciona un número con un carácter. Por ejemplo, en la tabla ASCII, el número 65 corresponde a la letra "A".

Si el archivo que estamos leyendo fue guardado con un encoding distinto al que pandas espera, nos arrojará un error. 

Para solucionar esto, podemos utilizar el parámetro `encoding` de la función `pd.read_csv()` y especificar el encoding correcto.

¿Pero cómo sabemos con qué encoding cuenta el archivo?

La realidad es que la gran mayoría de los archivos los encontrarán en encoding utf-8 y Pandas no va a dar ningún error. Aquí estamos teniendo este problema porque la computadora que utilizan para generar este archivo uso un encoding diferente a utf-8.

En México, por lo general, si un archivo no está en utf-8, lo más seguro es que esté en `ISO-8859-1` o `latin1`.

Intentemos cargar el archivo especificando el encoding `ISO-8859-1`.

In [16]:
df = pd.read_csv('data/datos_delitos.csv', encoding='ISO-8859-1') 
df = limpiar_columnas(df) 
df.head(2)
                 

,anio,clave_ent,entidad,bien_juridico_afectado,tipo_de_delito,subtipo_de_delito,modalidad,enero,febrero,marzo,abril,mayo,junio,julio,agosto,septiembre,octubre,noviembre,diciembre
0,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,3,0,2,1,1,1,2.0,1.0,2.0,2.0,2.0,1.0
1,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0.0,1.0,0.0,0.0,0.0,1.0


Y vemos que ya podemos leer correctamente el archivo.


¿Existe alguna forma de verificar el encoding de un archivo sin tener que estar adivinando?

ChatGTP generó el siguiente código:

In [18]:
import chardet

def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        rawdata = f.read()
    result = chardet.detect(rawdata)
    return result

file_path = './data/datos_delitos.csv'
encoding_info = detect_encoding(file_path)
print(f"Detected encoding: {encoding_info['encoding']}")

Detected encoding: Windows-1252


In [17]:
%pip install chardet

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 19.0 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Sigamos...

In [4]:
df.tail(3)

,Año,Clave_Ent,Entidad,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
31357,2024,32,Zacatecas,Otros bienes jurídicos afectados (del fuero co...,Delitos cometidos por servidores públicos,Delitos cometidos por servidores públicos,Delitos cometidos por servidores públicos,20,22,34,35,34,37,NaN,NaN,NaN,NaN,NaN,NaN
31358,2024,32,Zacatecas,Otros bienes jurídicos afectados (del fuero co...,Electorales,Electorales,Electorales,0,1,0,11,31,23,NaN,NaN,NaN,NaN,NaN,NaN
31359,2024,32,Zacatecas,Otros bienes jurídicos afectados (del fuero co...,Otros delitos del Fuero Común,Otros delitos del Fuero Común,Otros delitos del Fuero Común,151,183,174,199,208,177,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 31360 entries, 0 to 31359
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   anio                    31360 non-null  int64  
 1   clave_ent               31360 non-null  int64  
 2   entidad                 31360 non-null  str    
 3   bien_juridico_afectado  31360 non-null  str    
 4   tipo_de_delito          31360 non-null  str    
 5   subtipo_de_delito       31360 non-null  str    
 6   modalidad               31360 non-null  str    
 7   enero                   31360 non-null  int64  
 8   febrero                 31360 non-null  int64  
 9   marzo                   31360 non-null  int64  
 10  abril                   31360 non-null  int64  
 11  mayo                    31360 non-null  int64  
 12  junio                   31360 non-null  int64  
 13  julio                   28224 non-null  float64
 14  agosto                  28224 non-null  float64
 

Bien. Están muy bien los datos. Sin embargo, tenemos un problema con el que es muy común encontrarnos. 

Resulta que el formato en el que está el archivo es fácil entender por seres humanos:


```markdown
| Año      | Entidad  | Enero    | Febrero  | Mes X    |
|----------|----------|----------|----------|----------|
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
```

Vemos que tenemos los meses como encabezados. Es decir, el archivo está tratando cada mes como si fuera una variable. 

Nosotros como científicos de datos estamos más interesados en conjuntos de datos que no estén en este formato de "resumen" o "tabla dinámica". Para nosotros, lo ideal sería que cada mes fuera simplemente una observación más en nuestro conjunto de datos. Es decir, queremos transformar la tabla de arriba en:

```markdown
| Año      | Entidad  | Mes        |
|----------|----------|------------|
|   ...    |   ...    |   Enero    |
|   ...    |   ...    |   Febrero  |
|   ...    |   ...    |   Marzo    |
|   ...    |   ...    |   Abril    |
|   ...    |   ...    |   Mes X    |
```

A este tipo de formato le llamos "formato largo de datos". 

Haremos las siguientes limpiezas:
* Transformar los nombres de las columnas para que no tengan caracteres especiales y estén siempre en minúsculas
* Convertir el dataset a un formato de datos "largo"

In [20]:
for col in df.columns:
    print(col)

anio
clave_ent
entidad
bien_juridico_afectado
tipo_de_delito
subtipo_de_delito
modalidad
enero
febrero
marzo
abril
mayo
junio
julio
agosto
septiembre
octubre
noviembre
diciembre


In [21]:
len(df.columns)

19

### Limpiar nombres de columnas

In [11]:
def limpiar_columnas(df):
    columnas_limpias = []
    for col in df.columns:
        # convertir a minusculas, reemplazar espacios por guiones bajos y eliminar caracteres especiales
        col = col.lower().replace(" ", "_").replace("ñ", "ni").replace(".", "").replace("á", "a").replace("é", "e").replace("í","i").replace("ó", "o").replace("ú", "u")
        columnas_limpias.append(col)
    
    df.columns = columnas_limpias
    
    return df


In [12]:
df.head(2)

,Año,Clave_Ent,Entidad,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
0,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,3,0,2,1,1,1,2.0,1.0,2.0,2.0,2.0,1.0
1,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0.0,1.0,0.0,0.0,0.0,1.0


In [ ]:
df = limpiar_columnas(df)

In [ ]:
df.head(3)

Muy bien. Ahora lo que queremos hacer es quitar algunas columnas. Nos interesan nada más las siguientes:

In [22]:
df[['anio', 'entidad']]

,anio,entidad
0,2015,Aguascalientes
1,2015,Aguascalientes
2,2015,Aguascalientes
3,2015,Aguascalientes
4,2015,Aguascalientes
...,...,...
31355,2024,Zacatecas
31356,2024,Zacatecas
31357,2024,Zacatecas
31358,2024,Zacatecas


In [23]:
df = df[['anio', 'clave_ent', 'entidad', 'tipo_de_delito', 'subtipo_de_delito', 'modalidad','enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio', 'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre']]
df.head(2)

,anio,clave_ent,entidad,tipo_de_delito,subtipo_de_delito,modalidad,enero,febrero,marzo,abril,mayo,junio,julio,agosto,septiembre,octubre,noviembre,diciembre
0,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con arma de fuego,3,0,2,1,1,1,2.0,1.0,2.0,2.0,2.0,1.0
1,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0.0,1.0,0.0,0.0,0.0,1.0


### Formato largo de datos

Ahora, usaremos el método `melt` para convertir las columnas a observaciones.

Queremos convervar las coumnas:
* anio
* clave_ent
* entidad
* tipo_de_delito
* subtipo_de_delito
* modalidad

El resto de las columnas las vamos a juntar en una nueva columna llamada "nombre_mes" y sus valores los vamos a sumar en otra llamada "frecuencia"

In [24]:
print("Shape ", df.shape)

Shape  (31360, 18)


In [25]:
datos_long = df.melt(id_vars=['anio', 'clave_ent', 'entidad','tipo_de_delito', 'subtipo_de_delito', 'modalidad'], var_name='nombre_mes', value_name='frecuencia')

In [26]:
print("Shape: ", datos_long.shape)

Shape:  (376320, 8)


In [27]:
datos_long.head(5)

,anio,clave_ent,entidad,tipo_de_delito,subtipo_de_delito,modalidad,nombre_mes,frecuencia
0,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con arma de fuego,enero,3.0
1,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con arma blanca,enero,1.0
2,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con otro elemento,enero,0.0
3,2015,1,Aguascalientes,Homicidio,Homicidio doloso,No especificado,enero,2.0
4,2015,1,Aguascalientes,Homicidio,Homicidio culposo,Con arma de fuego,enero,0.0


In [28]:
datos_long.info()

<class 'pandas.DataFrame'>
RangeIndex: 376320 entries, 0 to 376319
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   anio               376320 non-null  int64  
 1   clave_ent          376320 non-null  int64  
 2   entidad            376320 non-null  str    
 3   tipo_de_delito     376320 non-null  str    
 4   subtipo_de_delito  376320 non-null  str    
 5   modalidad          376320 non-null  str    
 6   nombre_mes         376320 non-null  str    
 7   frecuencia         357504 non-null  float64
dtypes: float64(1), int64(2), str(5)
memory usage: 23.0 MB


Supongamos que para este análisis, no nos importan los niveles subtipo de delito y modalidad. O sea, no queremos tener la distinción entre homicidios dolosos y culposos (sé que son bastante diferentes, pero simplifiquemos nuestro ejemplo).

Vamos a agrupar nuestro dataframe por anio, clave_ent, entidad, tipo_de_delito y nombre_mes. Esto hará que todos los tipos de homicidios se sumen al tipo "homicidio" o todos los tipos de robo de vehículo (con o sin violencia) se sumen a "robo de vehículo".

In [29]:
datos_long = datos_long.groupby(['anio', 'clave_ent', 'entidad', 'tipo_de_delito', 'nombre_mes'])['frecuencia'].sum().reset_index()

In [30]:
datos_long[datos_long.tipo_de_delito == 'Robo'].sample(15)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
70951,2019,20,Oaxaca,Robo,marzo,1046.0
56550,2018,22,Querétaro,Robo,junio,2295.0
138631,2024,1,Aguascalientes,Robo,marzo,938.0
58466,2018,26,Sonora,Robo,diciembre,600.0
42154,2017,24,San Luis Potosí,Robo,octubre,1077.0
125671,2023,6,Colima,Robo,marzo,567.0
36384,2017,12,Guerrero,Robo,abril,912.0
18154,2016,6,Colima,Robo,octubre,650.0
71908,2019,22,Querétaro,Robo,febrero,2208.0
90625,2020,29,Tlaxcala,Robo,agosto,265.0


Mostremos todos los estados y su respectiva clave

In [31]:
datos_long[['clave_ent', 'entidad']].drop_duplicates().sort_values('entidad')

,clave_ent,entidad
0,1,Aguascalientes
480,2,Baja California
960,3,Baja California Sur
1440,4,Campeche
2880,7,Chiapas
3360,8,Chihuahua
3840,9,Ciudad de México
1920,5,Coahuila de Zaragoza
2400,6,Colima
4320,10,Durango


Ahora Veamos todos los datos de delitos de una entidad en específico. Por ejemplo, Nuevo león

In [32]:
datos_long[datos_long['clave_ent'] == 19].sample(15)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
55029,2018,19,Nuevo León,Otros delitos contra la familia,noviembre,360.0
55051,2018,19,Nuevo León,Otros delitos del Fuero Común,marzo,258.0
85878,2020,19,Nuevo León,Violación equiparada,junio,30.0
146993,2024,19,Nuevo León,Delitos cometidos por servidores públicos,julio,0.0
24028,2016,19,Nuevo León,Abuso sexual,febrero,70.0
101085,2021,19,Nuevo León,Narcomenudeo,noviembre,329.0
147213,2024,19,Nuevo León,Otros delitos del Fuero Común,noviembre,0.0
70100,2019,19,Nuevo León,Abuso de confianza,mayo,80.0
85489,2020,19,Nuevo León,Allanamiento de morada,agosto,25.0
101272,2021,19,Nuevo León,Violencia familiar,febrero,1257.0


In [33]:
datos_long[(datos_long['clave_ent'] > 19) &  (datos_long['clave_ent'] < 24) & (datos_long['tipo_de_delito'] == 'Homicidio')].sample(15)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
9817,2015,21,Puebla,Homicidio,agosto,115.0
148539,2024,22,Querétaro,Homicidio,enero,30.0
101987,2021,21,Puebla,Homicidio,septiembre,92.0
41025,2017,22,Querétaro,Homicidio,noviembre,23.0
56864,2018,23,Quintana Roo,Homicidio,mayo,75.0
71745,2019,22,Querétaro,Homicidio,noviembre,46.0
117820,2022,22,Querétaro,Homicidio,febrero,37.0
87099,2020,22,Querétaro,Homicidio,enero,36.0
41507,2017,23,Quintana Roo,Homicidio,septiembre,44.0
41506,2017,23,Quintana Roo,Homicidio,octubre,59.0


### Valores de fechas

Finalmente, queremos tener una columna "fecha". Actualmente tenemos el año y el nombre del mes, pero no tenemos como tal una columna que tenga un tipo de dato fecha. Eso hace que filtrar por fecha sea complicado.

Por ejemplo, si queremos conocer todos los homicidios de Oaxaca en enero 2024, haríamos lo siguiente:

In [34]:
datos_long[
    (datos_long['clave_ent'] == 20) &
    (datos_long['tipo_de_delito'] == 'Homicidio') &
    (datos_long['anio'] == 2024) &
    (datos_long['nombre_mes'] == 'enero') 
]

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
147579,2024,20,Oaxaca,Homicidio,enero,157.0


Creemos una columna de fecha. 

Primero tenemos que convertir el nombre de mes a un número, en donde 1 es enero, 2 febrero, etc.

In [35]:
datos_long.sample(2)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
57547,2018,24,San Luis Potosí,Tráfico de menores,marzo,0.0
94943,2021,6,Colima,Rapto,septiembre,0.0


In [36]:
datos_long["nueva_columna"] = "dato vacío"
datos_long.sample(6)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna
53920,2018,17,Morelos,Extorsión,febrero,1.0,dato vacío
47493,2018,3,Baja California Sur,Violación simple,noviembre,15.0,dato vacío
92733,2021,2,Baja California,Corrupción de menores,noviembre,49.0,dato vacío
93771,2021,4,Campeche,Falsedad,enero,0.0,dato vacío
112239,2022,10,Durango,Secuestro,enero,0.0,dato vacío
111284,2022,8,Chihuahua,Secuestro,mayo,1.0,dato vacío


In [37]:
datos_long.drop(columns=['nueva_columna'], inplace=True)

In [38]:
# Diccionario de ayuda para convertir
meses = {
    "enero": 1,
    "febrero": 2,
    "marzo": 3,
    "abril": 4,
    "mayo": 5,
    "junio": 6,
    "julio": 7,
    "agosto": 8,
    "septiembre": 9,
    "octubre": 10,
    "noviembre": 11,
    "diciembre": 12
}

datos_long['mes'] = datos_long['nombre_mes'].map(meses)
datos_long.sample(5)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,mes
44080,2017,28,Tamaulipas,Secuestro,febrero,14.0,2
34769,2017,9,Ciudad de México,Fraude,julio,1051.0,7
122541,2022,32,Zacatecas,Electorales,noviembre,3.0,11
73754,2019,26,Sonora,Otros delitos contra la sociedad,diciembre,2.0,12
151946,2024,29,Tlaxcala,Lesiones,diciembre,0.0,12


In [39]:
datos_long["frecuencia_mas_10"] = datos_long["frecuencia"] + 10
datos_long.sample(5)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,mes,frecuencia_mas_10
88091,2020,24,San Luis Potosí,Incesto,septiembre,0.0,9,10.0
21241,2016,13,Hidalgo,Despojo,agosto,29.0,8,39.0
33638,2017,7,Chiapas,Acoso sexual,diciembre,0.0,12,10.0
113714,2022,13,Hidalgo,Violación equiparada,diciembre,28.0,12,38.0
82404,2020,12,Guerrero,Otros delitos del Fuero Común,abril,124.0,4,134.0


In [41]:
datos_long.drop(columns=['frecuencia_mas_10'], inplace=True)

KeyError: "['frecuencia_mas_10'] not found in axis"

In [42]:
datos_long["anio_mes"] = datos_long["anio"].astype(str) + datos_long["mes"].astype(str)
datos_long.sample(6)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,mes,anio_mes
89403,2020,27,Tabasco,Despojo,enero,43.0,1,20201
85694,2020,19,Nuevo León,Incumplimiento de obligaciones de asistencia f...,diciembre,35.0,12,202012
56600,2018,22,Querétaro,Violación equiparada,mayo,6.0,5,20185
81955,2020,11,Guanajuato,Otros delitos que atentan contra la libertad y...,marzo,2.0,3,20203
28714,2016,28,Tamaulipas,Robo,octubre,1379.0,10,201610
58931,2018,27,Tabasco,Otros delitos que atentan contra la vida y la ...,septiembre,33.0,9,20189


In [43]:
# yyyy-mm-dd, yy-mm-dd, yymmdd, yyyy/dd/mm

In [44]:
# Agregamos la columna de fecha juntando el año y el mes
datos_long['fecha'] = pd.to_datetime(datos_long['anio'].astype(str) + datos_long['mes'].astype(str), format='%Y%m')
datos_long.sample(5)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,mes,anio_mes,fecha
106687,2021,31,Yucatán,Despojo,marzo,0.0,3,20213,2021-03-01
3868,2015,9,Ciudad de México,Abuso sexual,febrero,102.0,2,20152,2015-02-01
109398,2022,4,Campeche,Violación equiparada,junio,23.0,6,20226,2022-06-01
70611,2019,20,Oaxaca,Allanamiento de morada,enero,24.0,1,20191,2019-01-01
78545,2020,4,Campeche,Otros delitos contra la familia,julio,0.0,7,20207,2020-07-01


In [46]:
# Eliminamos las columnas que ya no necesitamos
datos_long = datos_long.drop(columns=['nueva_columna'], errors='ignore')
datos_long.sample(5)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,mes,anio_mes,fecha
92203,2021,1,Aguascalientes,Acoso sexual,marzo,0.0,3,20213,2021-03-01
9279,2015,20,Oaxaca,Extorsión,enero,0.0,1,20151,2015-01-01
85361,2020,18,Nayarit,Secuestro,julio,0.0,7,20207,2020-07-01
95734,2021,8,Chihuahua,Fraude,octubre,288.0,10,202110,2021-10-01
52130,2018,13,Hidalgo,Otros delitos contra el patrimonio,diciembre,21.0,12,201812,2018-12-01


Veamos los homicidios en oaxaca de enero 2024 a la fecha

In [47]:
datos_long[
    (datos_long.tipo_de_delito == "Homicidio") &
    (datos_long.clave_ent == 20) &
    (datos_long.fecha >= '2024-01-01')
].sort_values('fecha')

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,mes,anio_mes,fecha
147579,2024,20,Oaxaca,Homicidio,enero,157.0,1,20241,2024-01-01
147580,2024,20,Oaxaca,Homicidio,febrero,170.0,2,20242,2024-02-01
147583,2024,20,Oaxaca,Homicidio,marzo,190.0,3,20243,2024-03-01
147576,2024,20,Oaxaca,Homicidio,abril,208.0,4,20244,2024-04-01
147584,2024,20,Oaxaca,Homicidio,mayo,236.0,5,20245,2024-05-01
147582,2024,20,Oaxaca,Homicidio,junio,202.0,6,20246,2024-06-01
147581,2024,20,Oaxaca,Homicidio,julio,0.0,7,20247,2024-07-01
147577,2024,20,Oaxaca,Homicidio,agosto,0.0,8,20248,2024-08-01
147587,2024,20,Oaxaca,Homicidio,septiembre,0.0,9,20249,2024-09-01
147586,2024,20,Oaxaca,Homicidio,octubre,0.0,10,202410,2024-10-01


In [48]:
datos_finales = datos_long[['anio', 'clave_ent', 'entidad', 'tipo_de_delito', 'nombre_mes', 'fecha', 'frecuencia']]
datos_finales.head(2)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,fecha,frecuencia
0,2015,1,Aguascalientes,Aborto,abril,2015-04-01,0.0
1,2015,1,Aguascalientes,Aborto,agosto,2015-08-01,0.0


Ya que tenemos muestros datos bien estructurados, los podemos guardar en nuestra computadora. Los guardaremos con el nombre "delitos.csv"

In [50]:
datos_finales.to_csv('data/delitos.csv', index=False)